In [1]:
import pickle
from xml.parsers.expat import model
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.metrics import (
    auc,
    roc_auc_score,
    precision_score,
    recall_score,
    roc_curve,
    precision_recall_curve,
    classification_report,
)
from EGraphSAGE import EGraphSAGE
import sys

ModuleNotFoundError: No module named 'EGraphSAGE'

In [ ]:
test_flows = pd.read_csv('interm/ToNIoT_processed_train.csv')

In [ ]:
run_dir = 'interm/runs/EGraphSAGE_anomdetection_ToNIoT_graphsage_20260224_172938'

In [ ]:
def get_preds(exp_dir, test_flows_df, device='cpu'):
    print(f"Experiment directory: {exp_dir}")
    figure = Path("figures") / exp_dir.name
    figure.mkdir(exist_ok=True)

    # Load experiment metadata
    with open(exp_dir / "experiment.pkl", "rb") as f:
        cfg = pickle.load(f)

    WINDOW = cfg["window_size"]

    # Rebuild model
    with open(exp_dir / "best_model.pkl", "rb") as f:
        model = pickle.load(f)
    model.to(device)
    model.eval()

    attack_labels = test_flows_df["Attack"].values  # original string labels

    # Binary encoding (same logic as training)
    test_flows_df["Attack"] = torch.Tensor(
        (test_flows_df["Attack"] != "Benign").astype(float)
    ).float()

    # Dummy criterion (not used in eval)
    criterion = torch.nn.BCEWithLogitsLoss()

    with torch.no_grad():
        _, y_true_bin, y_probs, _ = model.train_flows(
            test_flows,
            criterion=criterion,
            optimizer=None,
            window=WINDOW,
            train=False,
        )

    return y_true_bin, y_probs

In [ ]:
y_true, y_probs = get_preds()